<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>11. Operaciones sobre grupos</font>  </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

En esta libreta se trabajará con una versión del conjunto de datos `titanic.csv` en la que se han cambiado los nombres de las columnas y se han recodificado algunos valores.  

In [ ]:
import pandas as pd
import numpy as np

df_titanic = pd.read_csv('data/titanic_prep.csv', index_col=0)
df_titanic.head()

---

## <font color="#5E5E5E">1. Agregación por grupos</font>

Algunas medidas de agregación, como por ejemplo la media, actúan sobre cada columna de un `DataFrame`. En el caso de una agrupación, actúan sobre cada columna ***para cada uno de los grupos***, y devuelven un `DataFrame` (o una `Series` si se hacen sobre una sola columna seleccionada con un escalar).

In [ ]:
df_titanic.groupby('Clase')[['Tarifa','Edad']].mean(numeric_only=True)

In [ ]:
df_titanic.groupby('Clase')['Tarifa'].agg(['mean', "std"])

También pueden aplicarse sobre varias columnas (sobre un `DataFrame`). En ese caso, se crea un índice de dos niveles para las columnas.

In [ ]:
df_titanic.groupby('Clase')[['Tarifa','Edad']].agg(['mean', 'std'])

En el siguiente ejemplo se combinan tanto la agrupación por dos columnas, como el acceso a otras dos columnas, y el uso de dos funciones de agregación. El resultado es un `DataFrame` con dos índices (filas / columnas) de dos niveles.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Al encerrar la sentencia entre paréntesis, se puede escribir en varias líneas. Este tipo de codificación ayuda a organizar y entender el código.

</div>

In [ ]:
(df_titanic.groupby(['Puerto','Clase'])
           [['Tarifa','Edad']]
           .agg(['mean', 'std'])
)

Como es difícil trabajar con esta estructura, puede convenir dejar las claves de grupo como columnas mediante `as_index=False` y asignar los nombres a las columnas del resumen.

In [ ]:
summary = (df_titanic.groupby(['Puerto','Clase'], as_index=False)
                     [['Tarifa','Edad']]
                     .agg(['mean', 'std'])
                     .set_axis(['Puerto', 'Clase', 'Tarifa (media)', 'Tarifa (desv)', 'Edad (media)', 'Edad (desv)'], axis=1)
    )

summary

También es posible aplicar medidas de agregación distintas a cada columna. Para ello, se pasa al método `DataFrameGroupBy.agg()` un diccionario en el que las claves son nombres de columnas, y los valores son las funciones (o listas de funciones) que se aplican a cada columna.

In [ ]:
(df_titanic.groupby(['Puerto','Clase'], as_index=False)
           [['Tarifa','Edad']]
           .agg({
                'Tarifa'  :['min','max'],
                  'Edad'  :'mean'
                }
           )
)

También se permite nombrar las columnas resultantes de las agregaciones directamente. Existe una restricción, y es la necesidad de que los nombres sean compatibles con los identificadores en *python*.

In [ ]:
(df_titanic.groupby(['Puerto','Clase'], as_index=False)
           [['Tarifa','Edad']]
           .agg( 
                Max_tarifa=('Tarifa','max'),
                Min_tarifa=('Tarifa','min'),
                Edad_media=('Edad','mean')
            )
)

Existe una forma de utilizar directamente nombres de columnas no compatibles con identificadores de *python*.

In [ ]:
(df_titanic.groupby(['Puerto','Clase'], as_index=False)
           [['Tarifa','Edad']]
           .agg(**{ 
                "Tarifa máxima" : pd.NamedAgg(column='Tarifa',aggfunc='max'),
                "Tarifa mínima" : pd.NamedAgg(column='Tarifa',aggfunc='min'),
              })
)

O directamente con diccionarios en lugar de `pd.NamedAgg`

In [ ]:
(df_titanic.groupby(['Puerto', 'Clase'], as_index=False)
           .agg(**{
               'Tarifa máxima': ('Tarifa', 'max'),
               'Tarifa mínima': ('Tarifa', 'min')
           })
)


---

## <font color="#5E5E5E">2. Filtrado por grupos</font>


El método `DataFrameGroupBy.filter(condición)` permite filtrar las filas de un `DataFrame` en función de una condición definida a nivel de grupo.  Esta `condición` **debe ser una función que actúe sobre un `DataFrame`** (actuará sobre el correspondiente a cada grupo) y devuelva `True` o `False`, dependiendo si el grupo la cumple o no. Como resultado, `DataFrameGroupBy.filter()` devuelve **un  nuevo `DataFrame` con las filas correspondientes** a los grupos que cumplen la condición.

En este ejemplo, se crea una función `check_df`, que comprueba si el `DataFrame` tiene más de 200 filas. El resultado del filtrado es otro `DataFrame` con las filas del original que corresponden a grupos cuyo tamaño es mayor que 200. En este caso, se ha agrupado por el valor de la columna `Clase`, y devuelve las filas correspondientes a las clases `first` y `third`. 

In [ ]:
#df_titanic['Clase'].value_counts()

In [ ]:
def check_df(df_group):
    return len(df_group)>200

df_titanic.groupby('Clase').filter(check_df)

---

## <font color="#5E5E5E">3. Transformación en grupos</font>


En algunos casos, el valor de los elementos de la nueva `Serie` (columna) puede depender de una **transformación a partir de los datos de un grupo**. El método `DataFrameGroupBy.transform(función)` recibe una `función` que opera sobre un `DataFrame` o un `Series` (el correspondiente a cada grupo) y que puede devolver un valor, otra `Series`, u otro `DataFrame`. Como resultado, `DataFrameGroupBy.transform()` devuelve una `Series`/`DataFrame` con un **elemento/fila por cada fila del `DataFrame` original**.


Un ejemplo de esta situación es la normalización por grupo. Para ilustrarla, se recurrirá a un `DataFrame` con un resumen con la supervivencia por `Clase` y `Sexo`.

In [ ]:
# Crea un DataFrame para los ejemplos
df_titanic_sum = (df_titanic.groupby(['Clase','Sexo','Sobrevive'])
                            .size()
                            .reset_index()
                            .rename(columns={0: 'Personas'}))

df_titanic_sum.head(6) # 6 primeras filas, 3 primeros grupos

#### <font color="#5E5E5E"> La función de transformacion devuelve un valor</font>

Por ejemplo, a partir de `df_titanic_sum`, se quiere obtener la proporción de supervivientes y no supervivientes para cada categoría  `['Clase','Sexo']`. Esto no puede calcularse solo a partir de cada fila, sino que es necesario disponer del total de personas en cada categoría.  

Si se pasa a `DataFrameGroupBy.transform()` la función `sum`, suma los valores de los elementos del grupo. Éste valor puede ser almacenado en una nueva columna. 

In [ ]:
df_titanic_sum['Suma'] = df_titanic_sum.groupby(['Clase','Sexo'])['Personas'].transform("sum")
df_titanic_sum.head(6) # 6 primeras filas, 3 primeros grupos

Esta transformación devuelve una serie con el mismo valor para cada fila del grupo, lo que permite operar entre elementos y agregaciones por grupo, es decir, hacer una transformación por grupo. En este caso, para calcular la proporción.

In [ ]:
(df_titanic_sum['Personas']/df_titanic_sum['Suma']).head(6) 

# El 3.2% de las mujeres en primera clase no sobrevive
# El 96.8% de los hombres en primera clase no sobrevive

#### <font color="#5E5E5E"> La función de transformacion devuelve una *Serie*</font>

Es posible hacer la operación anterior dentro de la función de transformación, y que ésta devuelva directamente una `Series` con los valores correspondientes para cada fila. La función debe devolver una `Series` con tantos elementos como el `DataFrame` o `Serie` que se le pasa como argumento.

In [ ]:
# df_titanic_sum['Personas'] / df_titanic_sum['Personas'].sum()

def proporcion(s):
    return s/s.sum()

df_titanic_sum['Prop'] = df_titanic_sum.groupby(['Clase','Sexo'])['Personas'].transform(proporcion)
df_titanic_sum.head(6)

En operaciones sencillas, se puede utilizar directamente una función *lambda*.

In [ ]:
df_titanic_sum['Prop'] = df_titanic_sum.groupby(['Clase','Sexo'])['Personas'].transform(lambda s: s/s.sum())
df_titanic_sum.head(6)

#### <font color="#5E5E5E"> La función de transformacion devuelve un *DataFrame*</font>

Cuando la función que se pasa a `DataFrameGroupBy.transform()`, devuelve un `DataFrame`, éste debe tener tantas filas como el `DataFrame` original. 


En esta celda se crea un `DataFrame` ejemplo añadiendo la suma de las tarifas para cada categoría `df_titanic_sum[[Clase`,`Sexo]]` a `df_titanic_sum`.

***¡OJO!:*** Esto no es una transformación. Se ha creado un nuevo DataFrame para ilustrar el ejemplo.

In [ ]:
# Crea un DataFrame ejemplo
df_titanic_sum.drop(columns=['Suma','Prop'], inplace=True)
df_titanic_sum['Suma tarifas']=(df_titanic.groupby(['Clase','Sexo','Sobrevive'])['Tarifa'].sum().reset_index())['Tarifa']
df_titanic_sum.head(6)

La función de transformación, en este caso, recibe un `DataFrame` y devuelve una `Series` con la proporción para cada una de las columnas.

In [ ]:
# df_titanic_sum[['Personas','Suma tarifas']]/df_titanic_sum[['Personas','Suma tarifas']].sum()
df_titanic_sum.groupby(['Clase','Sexo'])[['Personas','Suma tarifas']].transform(lambda df: df/df.sum()).head(6)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Una forma muy práctica de predecir el efecto de una transformación (o de los métodos que se verán a continuación) consiste en aplicar la función pasada como argumento al propio `DataFrame`, comprobar su salida, y luego visualizar cómo sería esa salida concatenándose para cada grupo.
</div>

---

### <font color="#5E5E5E">4. `DataFrameGroupBy.apply()`</font>


El método `DataFrameGroupBy.apply(función)` permite aplicar una función a cada uno de los grupos. Esta `función` debe tomar como entrada un `DataFrame`, y puede devolver un escalar, un `Series`, o un `DataFrame`.  `DataFrameGroupBy.apply()` es un **método muy flexible**, y también permite llevar a cabo  agregaciones y transformaciones a nivel de grupo. 




<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
**Importante.** Cuando se puede utilizar `DataFrameGroupBy.agg()` o `DataFrameGroupBy.transform()`,  **por eficiencia y claridad, se recomienda evitar el uso de** `DataFrameGroupBy.apply()`.
</div>

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Desde pandas 3.0, `DataFrameGroupBy.apply()` pasa a la función los datos del grupo **sin las columnas utilizadas para agrupar**. `include_groups=False` lo deja explícito y también funciona en pandas 2.2. Las claves siguen disponibles en el índice del resultado o como nombre del grupo. `include_groups=True` ya no está permitido en pandas 3.0.
</div>

Dependiendo de la salida de la función pasada como argumento, `DataFrameGroupBy.apply()` devuelve un tipo u otro de estructura.

#### <font color="#5E5E5E"> La función devuelve un escalar</font>

Por ejemplo, si la función devuelve un escalar, `DataFrameGroupBy.apply()` devuelve una serie con el valor agregado para cada grupo.

In [ ]:
df_titanic

In [ ]:
df_titanic.groupby('Clase').apply(lambda data: len(data)) # Agregación con el tamaño de cada grupo

#### <font color="#5E5E5E"> La función devuelve una `Series`</font>

Si la salida es un `Series`, depende de si el índice de ésta corresponde a las filas o las columnas del `DataFrame` original. En este caso, por ejemplo,  devuelve la media de cada columna.  Por tanto, el código devuelve la agregación para cada columna para cada grupo.

In [ ]:
df_titanic.groupby('Clase')[['Tarifa','Edad']].apply(lambda data: data.mean()) # Agregación con la media para cada grupo/columna
#df_titanic.groupby('Clase')[['Tarifa','Edad']].apply(lambda data: data.mean(axis=0)) # Equivalente

Si la salida de la función es un `Series` cuyo índice es el índice del `DataFrame`, `DataFrameGroupBy.apply()` devuelve un `Series` del tamaño del `DataFrame` original. Esta funcionalidad se puede utilizar para calcular valores que dependen del grupo concreto, es decir, para hacer transformaciones. 

En este ejemplo, para cada pasajero se calcula el porcentaje del total correspondiente a cada clase que supone su tarifa. El `Series` resultante tiene un índice de dos niveles (se verá después).

In [ ]:
def pct_val_fare(data):
    return 100*data['Tarifa']/data['Tarifa'].sum()

In [ ]:
# pct_val_fare(df_titanic).head()
df_titanic.groupby('Clase').apply(pct_val_fare).tail(20)  # Hace una transformación, aunque devuelve el índice de dos niveles.

La función también puede transformar el `DataFrame` original, añadiendo una columna con el porcentaje del total para cada clase que supone su tarifa.

In [ ]:
def pct_val_fare(data):
    data['% Tarifa'] = 100*data['Tarifa']/data['Tarifa'].sum()
    return data
    
df_titanic.groupby('Clase').apply(pct_val_fare)   # Hace una transformación directamente en el *DataFrame*

#### <font color="#5E5E5E"> La función devuelve un *DataFrame*</font>


En este ejemplo, la función devuelve el máximo y el mínimo para cada una de las columnas del `DataFrame` en otro `DataFrame`. Al utilizarla con `DataFrameGroupBy.apply()`, trata cada grupo por separado. El `DataFrame` resultante tiene un índice de dos niveles.

In [ ]:
def max_min(data):
    return data.select_dtypes(include=np.number).agg(["min", "max"])

# max_min(df_titanic)
df_titanic.groupby('Clase').apply(max_min, include_groups=False).head(10)

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>